## Dynamic Travel Concierge (Assistance)

### REAL API's

| Feature            | API                                              |
| ------------------ | ------------------------------------------------ |
| Weather            | OpenWeatherMap                                 |
| Places/Attractions | Google Places API (optional)                     |
| Hotels             | Simulated or RapidAPI (optional)                 |
| Flights            | Simulated (real flight APIs require paid access) |


------

Step 1 — Install Dependencies

-----


In [35]:
!pip install langchain langgraph langchain-google-genai faiss-cpu streamlit langchain_community
!pip install requests
!pip install amadeus
!pip install langchain-google-genai

------

Step 2 — Set API Key(TEMPORARY)

-------

In [36]:
#secret
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
AMADEUS_API_KEY = userdata.get("AMADEUS_API_KEY")
AMADEUS_API_SECRET = userdata.get("AMADEUS_API_SECRET")
OPENWEATHER_API_KEY = userdata.get("OPENWEATHER_API_KEY")


In [37]:
from amadeus import Client

amadeus = Client(
    client_id=AMADEUS_API_KEY,
    client_secret=AMADEUS_API_SECRET)

OPENWEATHER_API_KEY = OPENWEATHER_API_KEY


In [38]:
def get_city_code(city):
    try:
        response = amadeus.reference_data.locations.get(
            keyword=city,
            subType="CITY"
        )
        return response.data[0]["iataCode"]
    except:
        return None


------

Step 3 — Load Model

------

In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.4,google_api_key=GOOGLE_API_KEY)
#gemini-1.5-flash,gemini-2.5-flash-lite

-------

Step 4 — Define Travel Tools


-------

--- **WEATHER TOOL**

In [40]:
from langchain_core.tools import tool
import requests
from datetime import date
from datetime import timedelta

@tool
def get_weather(destination: str, date: date) -> str:
    """Fetches weather forecast for a given destination and date."""

    url = "http://api.openweathermap.org/data/2.5/forecast"
    params = {
        "q": destination,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric"
    }

    response = requests.get(url, params=params).json()

    if "list" not in response:
        return "Weather data unavailable"

    # Match forecast date
    target = str(date)
    for entry in response["list"]:
        if entry["dt_txt"].startswith(target):
            weather = entry["weather"][0]["main"]
            return weather

    return "Weather data unavailable"


--- **FLIGHT TOOL**

In [41]:

@tool
def search_flights(source: str, destination: str) -> str:
    """Search flights using Amadeus API with prices in INR."""

    try:
        origin_code = get_city_code(source.title())
        dest_code = get_city_code(destination.title())

        if not origin_code or not dest_code:
            return "Flight data unavailable."

        response = amadeus.shopping.flight_offers_search.get(
            originLocationCode=origin_code,
            destinationLocationCode=dest_code,
            departureDate=str(date.today() + timedelta(days=7)),
            adults=1,
            max=3,
            currencyCode="INR"
        )

        flights = []

        for offer in response.data:
            airline = offer["validatingAirlineCodes"][0]
            price = offer["price"]["total"]
            flights.append(f"{airline} ₹{price}")

        if not flights:
            return f"No flights found from {origin_code} to {dest_code}"

        return f"Flights {origin_code} → {dest_code}: " + " | ".join(flights)

    except Exception as e:
        return f"Flight API error: {str(e)}"

--- **HOTEL TOOL**

In [42]:
@tool
def search_hotels(destination: str) -> str:
    """Search hotels using Amadeus API."""

    try:
        city_code = get_city_code(destination)

        hotels = amadeus.reference_data.locations.hotels.by_city.get(
            cityCode=city_code
        )

        hotel_names = [hotel["name"] for hotel in hotels.data[:5]]

        return ", ".join(hotel_names).upper() if hotel_names else "No hotels found"

    except Exception as e:
        return "NA"



## 3b. RAG for Indoor Activities (Rain Handling)




In [43]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv("indoor_activities.csv", encoding='latin1')


if 'city,activity' in df.columns:
    df[['city', 'activity']] = df['city,activity'].str.split(',', n=1, expand=True)
    df = df.drop(columns=['city,activity'])
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Convert to LangChain Documents
indoor_docs = [
    Document(
        page_content=row["activity"],
        metadata={"city": row["city"].lower()}
    )
    for _, row in df.iterrows()
]

-----------

Step 2 — Create Vector Store

-----------------


In [44]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings
from langchain_core.documents import Document

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY)
docs = indoor_docs
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [45]:
"""
#
vectorstore = FAISS.from_documents(docs, FakeEmbeddings(size=32))

"""

'\n#\nvectorstore = FAISS.from_documents(docs, FakeEmbeddings(size=32))\n\n'

------

Step 3 — RAG Retrieval Function

-------


In [46]:
def get_indoor_activity(city: str):
    """Retrieve indoor activities using RAG with city → global fallback"""

    city = city.lower()

    docs = retriever.invoke(f"indoor activities in {city}")

    city_results = []
    global_results = []

    for doc in docs:
        doc_city = doc.metadata.get("city", "").lower()

        if doc_city == city:
            city_results.append(doc.page_content)

        elif doc_city == "global":
            global_results.append(doc.page_content)

    # Remove duplicates
    city_results = list(dict.fromkeys(city_results))
    global_results = list(dict.fromkeys(global_results))

    if city_results:
        return city_results

    if global_results:
        return global_results

    return [
        "Visit a museum",
        "Relax at a café",
        "Watch a movie indoors"
    ]

## 3c. State & Graph Implementation (LangGraph)

-----------

Step 1 — Define State

--------------

In [49]:
from typing import TypedDict, List


class TravelState(TypedDict):
    source: str
    destination: str
    start_date: str
    days: int
    itinerary: List[str]
    weather: List[str]
    flights: str
    hotels: str




In [50]:
from langchain_core.prompts import PromptTemplate

planner_prompt = PromptTemplate(
    input_variables=["destination", "days"],
    template="""
You are an expert travel planner creating a friendly and realistic itinerary.

Create a {days}-day travel plan for {destination}.

Planning rules:
- Each day should include 2–3 activities.
- If an attraction requires a full day, include only one.
- Group nearby places to reduce travel time.
- Balance sightseeing, culture, food, and relaxation.
- If weather is rainy, prefer indoor activities and day should include 2–3 activities
- Use natural, friendly language.

Output format:
Day 1:
- activity
- activity
- activity

Day 2:
- activity
- activity

Rules:
- Use bullet points (-)
- No extra commentary
- Keep each bullet under 15 words
- Exactly {days} days
"""
)

-----------

Step 2 — Planner Node

-------------

In [51]:
def planner_node(state: TravelState) -> TravelState:
    destination = state["destination"]
    days = state["days"]

    prompt = planner_prompt.format(destination=destination, days=days)
    response = llm.invoke(prompt).content.strip()

    lines = [line.strip() for line in response.split("\n") if line.strip()]

    itinerary = []
    current_day = ""

    for line in lines:
        if line.lower().startswith("day"):
            if current_day:
                itinerary.append(current_day.strip())
            current_day = line
        else:
            current_day += f" {line}"

    if current_day:
        itinerary.append(current_day.strip())


    while len(itinerary) < days:
        itinerary.append(f"Day {len(itinerary)+1}: Explore {destination}")

    state["itinerary"] = itinerary
    return state

-----------

Step 3 — Weather Node

----------

In [52]:
def weather_node(state: TravelState) -> TravelState:
    weather_report = []
    start_date = state["start_date"]
    days = state["days"]

    for day_offset in range(days):
        current_date = start_date + timedelta(days=day_offset)
        weather = get_weather.invoke({
            "destination": state["destination"],
            "date": current_date
        })
        weather_report.append(weather)

    state["weather"] = weather_report
    return state


--------------

Step 4 — Adjustment Node (Rain Handling + RAG)

-------------

In [53]:
def adjust_node(state: TravelState) -> TravelState:
    destination = state["destination"]

    indoor_options = get_indoor_activity(destination)

    updated_itinerary = []

    indoor_index = 0   # Track which indoor activity to use

    for day_index, (activity, weather) in enumerate(zip(state["itinerary"], state["weather"])):

        weather_lower = weather.lower()

        if "rain" in weather_lower:
            label = "Rain"
        elif "cloud" in weather_lower:
            label = "Clouds"
        elif "clear" in weather_lower or "sun" in weather_lower:
            label = "Clear"
        else:
            label = "NA"

        activity_text = activity.split(":", 1)[-1].strip()

        # Rain → choose next indoor activity
        if label == "Rain":

            if indoor_options:
                indoor = indoor_options[indoor_index % len(indoor_options)]
                indoor_index += 1
            else:
                indoor = f"Indoor activities in {destination.title()}"

            final_text = indoor

        else:
            final_text = activity_text

        updated_itinerary.append(f"Day {day_index+1}: {label}: {final_text}")

    state["itinerary"] = updated_itinerary

    return state

--------------------------------

Step 5 — Flight Node & Step 6 — Hotel Node

---------------------------

In [55]:


def flight_node(state):
    state["flights"] = search_flights.invoke({
        "source": state["source"],
        "destination": state["destination"]
    })
    return state


def hotel_node(state):
    state["hotels"] = search_hotels.invoke({
        "destination": state["destination"]
    })
    return state




---------------------

STEP 7 - Add finalize_node (for structured output)

--------------------

In [56]:


def finalize_node(state: TravelState) -> dict:
    return {
        "source": state["source"].title(),
        "destination": state["destination"].title(),
        "days": state["days"],
        "start_date": str(state["start_date"]),
        "itinerary": state["itinerary"],
        "Recommended Flights": state.get("flights", "NA"),
        "Recommended Hotels": state.get("hotels", "NA")
    }



------------------------

Step 8 — Build LangGraph

----------------



In [57]:
from langgraph.graph import StateGraph

graph = StateGraph(TravelState)

# Add nodes
graph.add_node("planner", planner_node)
graph.add_node("weather", weather_node)
graph.add_node("adjust", adjust_node)

graph.add_node("flights", flight_node)
graph.add_node("hotels", hotel_node)
graph.add_node("finalize", finalize_node)

# Entry point
graph.set_entry_point("planner")

# Flow
graph.add_edge("planner", "weather")
graph.add_edge("weather", "adjust")
graph.add_edge("adjust", "flights")
graph.add_edge("flights", "hotels")
graph.add_edge("hotels", "finalize")


# Compile
app = graph.compile()


------------------

Step 9 — Run the Agent

--------

In [59]:
result = app.invoke({
    "source": "Hyderabad",
    "destination": "indonesia",
    "start_date": date.today(),
    "days": 3
})
result

{'source': 'Hyderabad',
 'destination': 'Indonesia',
 'start_date': '2026-03-06',
 'days': 3,
 'itinerary': ['Day 1: Rain: Indoor theme park at Trans Studio',
  "Day 2: Clouds: - Wander through the Ujung Pandang market - Discover Fort Rotterdam's history - Savor local seafood by the harbor",
  'Day 3: Clouds: - Relax on Kuta Beach - Learn to surf with a lesson - Experience a Balinese massage'],
 'weather': ['Rain', 'Clouds', 'Clouds'],
 'flights': 'Flight data unavailable.',
 'hotels': 'NA'}